<a href="https://colab.research.google.com/github/Mogi21/pytorch-deep-learning-labs/blob/main/04_pytorch_cnns_cifar10.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
import os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from typing import Tuple, List, Dict

import torch
import torch.nn as nn
import torch.nn.functional as F
import torch.optim as optim
import torchvision
import torchvision.transforms as transforms
from torch.utils.data import Dataset, DataLoader

# ==========================================
# 1. CONFIGURATION & DEVICE SETUP
# ==========================================
# Ensures reproducibility
torch.manual_seed(42)
np.random.seed(42)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Executing on device: {device}")

BATCH_SIZE = 32
EPOCHS = 8
LEARNING_RATE = 1e-3

# ==========================================
# 2. DATA PIPELINE
# ==========================================
def get_cifar10_dataloaders() -> Tuple[DataLoader, DataLoader, DataLoader, tuple]:
    """Downloads and prepares CIFAR-10 dataloaders with standard normalization."""
    transform = transforms.Compose([
        transforms.ToTensor(),
        transforms.Normalize((0.5, 0.5, 0.5), (0.5, 0.5, 0.5)),
    ])

    trainset_full = torchvision.datasets.CIFAR10(root="./data", train=True, download=True, transform=transform)
    testset = torchvision.datasets.CIFAR10(root="./data", train=False, download=True, transform=transform)

    # Deterministic 80/20 train/val split
    gen = torch.Generator().manual_seed(42)
    train_size = int(0.8 * len(trainset_full))
    val_size = len(trainset_full) - train_size
    trainset, valset = torch.utils.data.random_split(trainset_full, [train_size, val_size], generator=gen)

    trainloader = DataLoader(trainset, batch_size=BATCH_SIZE, shuffle=True, num_workers=2)
    valloader = DataLoader(valset, batch_size=BATCH_SIZE, shuffle=False, num_workers=2)
    testloader = DataLoader(testset, batch_size=BATCH_SIZE, shuffle=False, num_workers=2)

    classes = ("plane", "car", "bird", "cat", "deer", "dog", "frog", "horse", "ship", "truck")

    return trainloader, valloader, testloader, classes

trainloader, valloader, testloader, classes = get_cifar10_dataloaders()

# ==========================================
# 3. UTILITIES & ARCHITECTURE
# ==========================================
def compute_output_shape(input_shape: Tuple[int, int, int], layers: List[tuple]) -> Tuple[int, int, int]:
    """
    Programmatically calculates the output tensor shape of a CNN without instantiating PyTorch modules.
    Demonstrates architectural shape tracking.
    """
    C, H, W = input_shape
    for layer in layers:
        if layer[0] == 'conv':
            _, out_channels, kernel_size, stride, padding = layer
            C = out_channels
            H = (H + 2 * padding - kernel_size) // stride + 1
            W = (W + 2 * padding - kernel_size) // stride + 1
        elif layer[0] == 'pool':
            _, kernel_size, stride = layer
            padding = 0
            H = (H + 2 * padding - kernel_size) // stride + 1
            W = (W + 2 * padding - kernel_size) // stride + 1
    return (C, H, W)

class CustomCNN(nn.Module):
    """
    A custom Convolutional Neural Network designed for 32x32 RGB image classification.
    Implements feature extraction blocks and fully connected classification layers with dropout regularization.
    """
    def __init__(self, num_classes: int = 10, dropout_rate: float = 0.3):
        super().__init__()
        # Feature Extraction Block 1
        self.conv1 = nn.Conv2d(3, 32, kernel_size=3, padding=1)
        self.bn1 = nn.BatchNorm2d(32)
        self.pool1 = nn.MaxPool2d(kernel_size=2, stride=2)

        # Feature Extraction Block 2
        self.conv2 = nn.Conv2d(32, 64, kernel_size=3, padding=1)
        self.bn2 = nn.BatchNorm2d(64)
        self.pool2 = nn.MaxPool2d(kernel_size=2, stride=2)

        # Classification Head
        self.fc1 = nn.Linear(64 * 8 * 8, 128)
        self.dropout = nn.Dropout(p=dropout_rate)
        self.fc2 = nn.Linear(128, num_classes)

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        x = self.pool1(F.relu(self.bn1(self.conv1(x))))
        x = self.pool2(F.relu(self.bn2(self.conv2(x))))
        x = x.flatten(1)
        x = F.relu(self.fc1(x))
        x = self.dropout(x)
        x = self.fc2(x) # Return raw logits for CrossEntropyLoss
        return x

# ==========================================
# 4. TRAINING ENGINE
# ==========================================
class ModelTrainer:
    """Handles the training and evaluation loops for PyTorch models."""
    def __init__(self, model: nn.Module, device: torch.device):
        self.model = model.to(device)
        self.device = device
        self.criterion = nn.CrossEntropyLoss()

    def evaluate(self, loader: DataLoader) -> Tuple[float, float]:
        """Calculates loss and accuracy on a given dataloader."""
        self.model.eval()
        total_loss, correct, total = 0.0, 0, 0
        with torch.no_grad():
            for x, y in loader:
                x, y = x.to(self.device), y.to(self.device)
                logits = self.model(x)
                loss = self.criterion(logits, y)
                total_loss += loss.item() * x.size(0)
                correct += (logits.argmax(1) == y).sum().item()
                total += y.size(0)
        return total_loss / total, correct / total

    def train(self, trainloader: DataLoader, valloader: DataLoader, epochs: int, lr: float) -> Dict[str, list]:
        """Executes the training loop and tracks metrics."""
        optimizer = optim.Adam(self.model.parameters(), lr=lr)
        history = {"train_loss": [], "val_loss": [], "val_acc": []}

        print(f"Beginning training for {epochs} epochs...")
        for epoch in range(1, epochs + 1):
            self.model.train()
            running_loss, seen = 0.0, 0

            for x, y in trainloader:
                x, y = x.to(self.device), y.to(self.device)
                optimizer.zero_grad()
                logits = self.model(x)
                loss = self.criterion(logits, y)
                loss.backward()
                optimizer.step()

                running_loss += loss.item() * x.size(0)
                seen += x.size(0)

            train_loss = running_loss / seen
            val_loss, val_acc = self.evaluate(valloader)

            history["train_loss"].append(train_loss)
            history["val_loss"].append(val_loss)
            history["val_acc"].append(val_acc)

            print(f"Epoch {epoch:02d} | Train Loss: {train_loss:.4f} | Val Loss: {val_loss:.4f} | Val Acc: {val_acc:.4f}")

        return history

# Train Custom CNN
custom_model = CustomCNN()
trainer = ModelTrainer(custom_model, device)
history = trainer.train(trainloader, valloader, epochs=EPOCHS, lr=LEARNING_RATE)

test_loss, test_acc = trainer.evaluate(testloader)
print(f"Custom CNN Test Accuracy: {test_acc:.4f}")

# ==========================================
# 5. TRANSFER LEARNING (RESNET18)
# ==========================================
print("\n--- Initiating Transfer Learning Pipeline ---")
# Pretrained models require standard ImageNet resizing and normalization
transform_resnet = transforms.Compose([
    transforms.Resize(64),
    transforms.ToTensor(),
    transforms.Normalize((0.5, 0.5, 0.5), (0.5, 0.5, 0.5)),
])

trainset_r_full = torchvision.datasets.CIFAR10(root="./data", train=True, download=False, transform=transform_resnet)
testset_r = torchvision.datasets.CIFAR10(root="./data", train=False, download=False, transform=transform_resnet)
gen = torch.Generator().manual_seed(42)
trainset_r, valset_r = torch.utils.data.random_split(trainset_r_full, [train_size, val_size], generator=gen)

trainloader_r = DataLoader(trainset_r, batch_size=128, shuffle=True, num_workers=2)
valloader_r = DataLoader(valset_r, batch_size=128, shuffle=False, num_workers=2)
testloader_r = DataLoader(testset_r, batch_size=128, shuffle=False, num_workers=2)

# Load pretrained ResNet18 and freeze backbone weights
resnet_model = torchvision.models.resnet18(weights=torchvision.models.ResNet18_Weights.DEFAULT)
for param in resnet_model.parameters():
    param.requires_grad = False

# Replace the classification head
in_features = resnet_model.fc.in_features
resnet_model.fc = nn.Linear(in_features, 10) # New head requires_grad=True by default

resnet_trainer = ModelTrainer(resnet_model, device)
resnet_history = resnet_trainer.train(trainloader_r, valloader_r, epochs=2, lr=LEARNING_RATE) # Fine-tune briefly

resnet_test_loss, resnet_test_acc = resnet_trainer.evaluate(testloader_r)
print(f"ResNet18 (Frozen Backbone) Test Accuracy: {resnet_test_acc:.4f}")

# ==========================================
# 6. INFERENCE ON CUSTOM DATASET
# ==========================================
# A custom Dataset class demonstrates the ability to ingest raw array data into PyTorch ecosystem
class NumpyImageDataset(Dataset):
    def __init__(self, data_array: np.ndarray, transform=None):
        # Reshape flat array (N, 3072) to (N, 3, 32, 32)
        self.data = data_array.reshape(-1, 3, 32, 32).astype(np.float32) / 255.0
        self.transform = transform

    def __len__(self):
        return len(self.data)

    def __getitem__(self, idx):
        x = self.data[idx]
        if self.transform:
            x = self.transform(torch.tensor(x))
        return x

if os.path.exists('test_data.npy'):
    print("\n--- Generating Predictions for Custom Test Set ---")
    data_test_raw = np.load('test_data.npy')

    # We apply the same normalization used during training
    normalize_transform = transforms.Normalize((0.5, 0.5, 0.5), (0.5, 0.5, 0.5))
    custom_testset = NumpyImageDataset(data_test_raw, transform=normalize_transform)
    custom_testloader = DataLoader(custom_testset, batch_size=64, shuffle=False)

    custom_model.eval()
    predictions = []

    with torch.no_grad():
        for x_batch in custom_testloader:
            x_batch = x_batch.to(device)
            logits = custom_model(x_batch)
            preds = logits.argmax(dim=1).cpu().numpy()
            predictions.extend(preds)

    pd.DataFrame(predictions, columns=['prediction']).to_csv('submission.csv', index=False)
    print("Inference complete. Results saved to 'submission.csv'.")
else:
    print("\n[Notice] 'test_data.npy' not found in working directory. Skipping custom inference step.")

Executing on device: cuda


 33%|███▎      | 56.9M/170M [20:19<57:32, 32.9kB/s]